## Medical Code Generator

Healthcare documentation often contains detailed doctor–patient conversations that must be converted into structured medical records for billing, clinical documentation, and analysis. 

Medical coders typically review these conversations and assign standardized codes such as ICD-10 (International Classification of Diseases) for diagnoses and HCPCS (Healthcare Common Procedure Coding System) for procedures and services. 

The goal of this project is to design and implement a retrieval-augmented LLM pipeline that extracts key medical information from clinical conversations, retrieves relevant codes from provided ICD-10 and HCPCS datasets, and outputs a structured response.

**Conversation → Medical Entity Extraction → Code Retrieval (Excel only) → LLM Validation → Structured Output**

**ICD-10** is a medical coding system used to classify diseases, conditions, symptoms, and causes of death. It is maintained by the World Health Organization.

It helps healthcare providers:
- Record diagnoses
- Track diseases and health conditions
- Maintain medical records
- Perform health statistics and research

**HCPCS** is a medical coding system used to describe medical services, procedures, equipment, and supplies. It is mainly used in medical billing and insurance claims.

It helps healthcare providers:
- Bill insurance companies
- Identify medical procedures
- Track medical services

#### 1. Handling files containing ICD-10 and HCPCS Codes

In [1]:
# read the ICD-10 and HCPCS code files
import pandas as pd
import openpyxl 
icd_df = pd.read_excel("ICD-10 Codes.xlsx")
hcpcs_df = pd.read_csv("HCPCS Codes.csv", encoding="latin1")

ICD-10 dataset

In [2]:
# inspect the dataframes
icd_df.head()

,CODE,SHORT DESCRIPTION (VALID ICD-10 FY2024),LONG DESCRIPTION (VALID ICD-10 FY2024)
0,A000,"Cholera due to Vibrio cholerae 01, biovar chol...","Cholera due to Vibrio cholerae 01, biovar chol..."
1,A001,"Cholera due to Vibrio cholerae 01, biovar eltor","Cholera due to Vibrio cholerae 01, biovar eltor"
2,A009,"Cholera, unspecified","Cholera, unspecified"
3,A0100,"Typhoid fever, unspecified","Typhoid fever, unspecified"
4,A0101,Typhoid meningitis,Typhoid meningitis


In [3]:
icd_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73201 entries, 0 to 73200
Data columns (total 3 columns):
 #   Column                                   Non-Null Count  Dtype
---  ------                                   --------------  -----
 0   CODE                                     73201 non-null  str  
 1   SHORT DESCRIPTION (VALID ICD-10 FY2024)  73201 non-null  str  
 2   LONG DESCRIPTION (VALID ICD-10 FY2024)   73199 non-null  str  
dtypes: str(3)
memory usage: 1.7 MB


In [4]:
# rename columns 
icd_df = icd_df.rename(columns={
    "CODE": "code",
    "SHORT DESCRIPTION (VALID ICD-10 FY2024)": "short_description",
    "LONG DESCRIPTION (VALID ICD-10 FY2024)": "long_description"
})

HCPCS Dataset

In [5]:
# inspect the dataframes
hcpcs_df.head()

,LIST OF CPT1/HCPCS CODES USED TO DEFINE CERTAIN DESIGNATED HEALTH SERVICE CATEGORIES2 UNDER SECTION 1877 OF THE SOCIAL SECURITY ACT,Unnamed: 1
0,NaN,NaN
1,"This code list is effective January 1, 2024",NaN
2,NaN,NaN
3,CLINICAL LABORATORY SERVICES,NaN
4,NaN,NaN


In [6]:
# drop rows with missing values
hcpcs_df.dropna(inplace=True)

In [7]:
hcpcs_df.info()

<class 'pandas.DataFrame'>
Index: 1211 entries, 7 to 1248
Data columns (total 2 columns):
 #   Column                                                                                                                               Non-Null Count  Dtype
---  ------                                                                                                                               --------------  -----
 0   LIST OF CPT1/HCPCS CODES USED TO DEFINE CERTAIN DESIGNATED HEALTH SERVICE CATEGORIES2 UNDER SECTION 1877 OF THE SOCIAL SECURITY ACT  1211 non-null   str  
 1   Unnamed: 1                                                                                                                           1211 non-null   str  
dtypes: str(2)
memory usage: 28.4 KB


In [8]:
# rename columns
hcpcs_df = hcpcs_df.rename(columns={
    "LIST OF CPT1/HCPCS CODES USED TO DEFINE CERTAIN DESIGNATED HEALTH SERVICE CATEGORIES2 UNDER SECTION 1877 OF THE SOCIAL SECURITY ACT": "code",
    "Unnamed: 1": "description"
})

#### 2. Convert DataFrames to LangChain Documents

In [9]:
from langchain_core.documents import Document

# document is the basic unit stored in vector database. 
# empty list stores all codes before sending them to vector store
documents = []

# ICD10
for _, row in icd_df.iterrows():
    documents.append(
        # iterate through each row and add the document containing code, description and metadata to the list
        Document(
            page_content=f"ICD10 Code: {row['code']}\nDescription: {row['short_description']} {row['long_description']}",
            metadata={
                "code": row["code"],
                "type": "ICD10"
            }
        )
    )

# HCPCS
for _, row in hcpcs_df.iterrows():
    documents.append(
        # iterate through each row and add the document containing code, description and metadata to the list
        Document(
            page_content=f"HCPCS Code: {row['code']}\nDescription: {row['description']}",
            metadata={
                "code": row["code"],
                "type": "HCPCS"
            }
        )
    )

print("Total documents:", len(documents))

Total documents: 74412


#### 3. Create embeddings to convert text into vectors

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

# embedding model is created to convert text data into vectors for semantic search
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### 4. Create Vector Database to store embeddings for retrieval

In [11]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 30,
        "fetch_k": 50
    }
)

#### 5. Defined Structured Output using Pydantic

In [12]:
from pydantic import BaseModel
from typing import List

class MedicalOutput(BaseModel):
    summary: str
    ICD10: List[str]
    HCPCS: List[str]

#### 6. Create LLM

In [13]:
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
from langchain.chat_models import init_chat_model
model = init_chat_model("google_genai:gemini-2.5-flash")
structured_model = model.with_structured_output(MedicalOutput)

#### 7. Create Output Parser

In [15]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=MedicalOutput)

#### 8. Create Prompts

In [43]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = SystemMessage(
content = f"""
You are an expert medical coder. Analyze the conversation and assign ONLY relevant ICD-10 diagnosis codes and HCPCS procedure codes.

CRITICAL RULES:
1. ALWAYS assign ICD-10 codes for ALL diagnoses mentioned (e.g., Type 2 diabetes → E11.*, hypertension → I10/I11.*).
2. Use ONLY codes from the retrieved documents below - check exact matches.
3. For procedures/exams (foot exam, labs), use HCPCS codes.
4. Do NOT leave ICD10 empty if diagnoses are present.
5. List multiple codes if multiple conditions.

Format EXACTLY as requested below.
{parser.get_format_instructions()}
""")

#### 9. Build Pipeline

In [44]:
def medical_coder(conversation):
    docs = retriever.invoke(conversation)
    context = "\n".join(
    [
        f"{d.metadata['type']} Code: {d.metadata['code']}\n{d.page_content}"
        for d in docs
    ]
)

    human_prompt = HumanMessage(
        content=f"""
            Conversation:{conversation}
            Relevant coding documents:{context}
            """
        )

    response = model.invoke([system_prompt, human_prompt])
    return parser.parse(response.content)

#### 10. Give input to LLM

In [45]:
conversation = open("case1.txt").read()
result = medical_coder(conversation)
print(result)

summary="The patient is a follow-up for chronic medical problems including Type 2 diabetes mellitus with diabetic polyneuropathy, dyslipidemia, and primary hypertension. The patient's medications (lisinopril, atorvastatin, metformin, glimepiride) are reviewed and continued. Blood pressure is slightly elevated, and home monitoring is advised. A diabetic foot exam was performed, revealing absent sensation to monofilament on both feet and hard skin on the right foot. Labs ordered include CBC, CMP, HbA1c, Vitamin B12, urine microalbumin, and a lipid panel. Due to the limitations of the provided coding documents, specific ICD-10 codes for the patient's chronic conditions could not be assigned as they were not present in the retrieved documents. Only the HCPCS code for the lipid panel was applicable from the provided list." ICD10=[] HCPCS=['80061']


In [46]:
result.summary

"The patient is a follow-up for chronic medical problems including Type 2 diabetes mellitus with diabetic polyneuropathy, dyslipidemia, and primary hypertension. The patient's medications (lisinopril, atorvastatin, metformin, glimepiride) are reviewed and continued. Blood pressure is slightly elevated, and home monitoring is advised. A diabetic foot exam was performed, revealing absent sensation to monofilament on both feet and hard skin on the right foot. Labs ordered include CBC, CMP, HbA1c, Vitamin B12, urine microalbumin, and a lipid panel. Due to the limitations of the provided coding documents, specific ICD-10 codes for the patient's chronic conditions could not be assigned as they were not present in the retrieved documents. Only the HCPCS code for the lipid panel was applicable from the provided list."

In [47]:
result.ICD10

[]

In [48]:
result.HCPCS

['80061']